# Task 1.1 — Core Contribution / Architecture (8 marks)

**Paper**: *Learning to Grade Short Answer Questions using Semantic Similarity Measures and Dependency Graph Alignments* (Mohler, Bunescu & Mihalcea, ACL 2011)

---

## Step-by-Step Method Description

### Step 1: Dependency Parsing of Answers
- **Description**: Both the instructor (reference) answer $A_i$ and the student answer $A_s$ are parsed using the Stanford Dependency Parser in collapse/propagate mode. The output is a set of `(relation, governor, dependent)` triples. The graphs are further post-processed to propagate dependencies across apposition relations, encode negation and POS tags within nodes, and add a ROOT node governing the main verb of each sentence.
- **Reference**: Section 3.1, Figure 1 (Pipeline model for scoring short-answer pairs)
- **Purpose**: To produce structured syntactic representations of answers that capture grammatical relationships (e.g., subject-verb-object), going beyond simple bag-of-words comparisons.

### Step 2: Node-to-Node Matching via Perceptron
- **Description**: For each node $x_i$ in the instructor's dependency graph and each node $x_s$ in the student's graph, a feature vector $\phi(x_i, x_s)$ of 68 features is computed. These include 36 semantic similarity features across four subgraph scopes ($N_x^0$ through $N_x^3$), plus 32 lexico-syntactic features (exact match, stemmed match, POS match, WordNet relations, role-based features). A linear scoring function $f(x_i, x_s) = w^T \phi(x_i, x_s)$ is learned using the averaged perceptron algorithm on manually annotated node-pair alignments (656 matches, 6647 non-matches from 32 student answers).
- **Reference**: Section 3.1, Table 1 (Perceptron Training Algorithm), Table 2 (68 features breakdown)
- **Purpose**: To learn a scoring function that determines how well individual nodes (words/subgraphs) in the student answer align with nodes in the reference answer, capturing both semantic and syntactic compatibility.

### Step 3: Graph-to-Graph Alignment via Hungarian Algorithm
- **Description**: The node-pair scores from Step 2 are used as edge weights in a bipartite graph, with instructor nodes on one side and student nodes on the other. Dummy nodes are added to handle unmatched nodes. The Hungarian algorithm finds the optimal one-to-one matching. Three transformations are applied: (1) normalization by the number of instructor nodes, (2) IDF weighting of instructor nodes, and (3) question demoting (removing question words from both answers). These 3 transformations create $2^3 = 8$ alignment scores assembled into feature vector $\psi_G(A_i, A_s)$.
- **Reference**: Section 3.2
- **Purpose**: To find the best overall alignment between the two answer graphs and produce alignment-based features that capture structural similarity.

### Step 4: Lexical Semantic Similarity (BOW Features)
- **Description**: Eleven bag-of-words (BOW) similarity measures are computed between each instructor-student answer pair: eight knowledge-based measures from WordNet (PATH, LCH, Lesk, WUP, RES, Lin, JCN, HSO) plus three corpus-based measures (LSA trained on Wikipedia CS articles, ESA on full Wikipedia, and TF-IDF cosine similarity). These are computed both with and without question demoting, producing feature vector $\psi_B(A_i, A_s)$ containing 11 semantic features.
- **Reference**: Section 3.3, Table 5
- **Purpose**: To capture word-level semantic relatedness that structural alignment alone may miss (e.g., synonyms, paraphrases), using both knowledge-based and distributional approaches.

### Step 5: Combined Grade Prediction via SVR / SVMRank
- **Description**: The alignment features $\psi_G(A_i, A_s)$ (8 features) and BOW features $\psi_B(A_i, A_s)$ (11 features, extended to ~22 with and without question demoting) are concatenated into a combined feature vector $\psi(A_i, A_s)$ of 30 total features. A grade $g(A_i, A_s) = u^T \psi(A_i, A_s)$ is computed using either SVR (minimizing MSE) or SVMRank (minimizing discordant pairs), both with linear kernels and parameters tuned via 5-fold grid search.
- **Reference**: Section 3.4, Table 7
- **Purpose**: To leverage supervised learning to optimally combine multiple heterogeneous features into a single grade prediction, improving upon using any individual measure in isolation.

### Step 6: Isotonic Regression for Score Calibration
- **Description**: The raw SVR/SVMRank output scores are transformed onto the original [0, 5] grading scale using isotonic regression, trained on a held-out fold. This is particularly important for SVMRank outputs which are relative ranking scores rather than absolute grades.
- **Reference**: Section 3.5
- **Purpose**: To ensure the final output is an interpretable grade on the same scale used by human annotators, enabling fair RMSE comparison.

---

## Final Summary

This paper addresses the problem of **automated short-answer grading** (assigning a continuous 0–5 grade to a student's free-text answer by comparing it to a reference answer), and the authors claim that their approach of **combining dependency-graph alignment features with multiple lexical semantic similarity measures using supervised learning (SVR/SVMRank)** outperforms using any individual similarity measure in isolation, as demonstrated by improved Pearson correlation and RMSE over standalone BOW baselines on a dataset of 2,273 student answers.